# MODIS–GNFS Fire Product Comparison: Reproducible Analysis Template

**Researcher:** Josephine Owusu  
**Study area:** Ghana  
**Study period:** 2020–2024  
**Tools:** Python, pandas, NumPy, Matplotlib, SciPy, scikit-learn

## Purpose
This notebook documents a reproducible workflow for preparing monthly regional fire-count tables, comparing satellite-derived MODIS fire counts with reference fire records, and calculating descriptive agreement statistics.

**Data protection:** The original research records are not included. Supply your own authorised input files locally. Do not commit confidential GNFS records, raw data, or outputs containing restricted information to this public repository.

## Expected input tables
- `gnfs_monthly.csv`: `Year`, `Month`, `Region`, `GNFS_Total`
- `modis_monthly.csv`: `Year`, `Month`, `Region`, `MODIS_Total`

The two files should contain monthly totals at the same regional and temporal scale. Check definitions, coverage, and missing months before interpreting comparisons.


## 1. Import libraries and define input paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_DIR = Path("data")
GNFS_FILE = DATA_DIR / "gnfs_monthly.csv"
MODIS_FILE = DATA_DIR / "modis_monthly.csv"


## 2. Load and standardise the monthly tables

In [ ]:
def standardise_monthly_table(df, count_column):
    """Standardise column names and basic types for a monthly regional table."""
    required = {"Year", "Month", "Region", count_column}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df[list(required)].copy()
    result["Year"] = pd.to_numeric(result["Year"], errors="coerce")
    result["Month"] = result["Month"].astype(str).str.strip().str.upper()
    result["Region"] = result["Region"].astype(str).str.strip().str.upper()
    result[count_column] = pd.to_numeric(result[count_column], errors="coerce")
    result = result.dropna(subset=["Year", "Month", "Region", count_column])
    result["Year"] = result["Year"].astype(int)
    return result

# Uncomment after placing authorised CSV files in the local data/ folder:
# gnfs = standardise_monthly_table(pd.read_csv(GNFS_FILE), "GNFS_Total")
# modis = standardise_monthly_table(pd.read_csv(MODIS_FILE), "MODIS_Total")


## 3. Merge by year, month, and region

In [ ]:
# Run after loading the two input tables above.
# comparison = pd.merge(
#     gnfs, modis,
#     on=["Year", "Month", "Region"],
#     how="inner",
#     validate="one_to_one"
# )
# print("Matched monthly-region records:", len(comparison))
# print(comparison.head())


## 4. National monthly aggregation

In [ ]:
# national = (
#     comparison.groupby(["Year", "Month"], as_index=False)
#     [["GNFS_Total", "MODIS_Total"]].sum()
# )
# national["Date"] = pd.to_datetime(
#     national["Year"].astype(str) + "-" + national["Month"],
#     format="%Y-%B", errors="coerce"
# )
# national = national.sort_values("Date")


## 5. Comparison statistics

Pearson and Spearman correlations describe association, while MAE, RMSE, and bias describe differences between paired counts. Correlation alone does **not** establish agreement or prove that one product accurately detects all fires. Interpret statistics alongside data coverage, reporting practices, spatial/temporal matching, and fire-detection limitations.

In [ ]:
def comparison_metrics(reference, satellite):
    """Return association and error metrics for paired numeric observations."""
    paired = pd.DataFrame({
        "reference": reference,
        "satellite": satellite
    }).dropna()

    if len(paired) < 2:
        raise ValueError("At least two paired observations are required.")

    x = paired["reference"].to_numpy()
    y = paired["satellite"].to_numpy()

    pearson_r = pearsonr(x, y) if np.std(x) > 0 and np.std(y) > 0 else (np.nan, np.nan)
    spearman_r = spearmanr(x, y) if np.std(x) > 0 and np.std(y) > 0 else (np.nan, np.nan)

    return {
        "n": len(paired),
        "pearson_r": pearson_r.statistic,
        "pearson_p": pearson_r.pvalue,
        "spearman_rho": spearman_r.statistic,
        "spearman_p": spearman_r.pvalue,
        "MAE": mean_absolute_error(x, y),
        "RMSE": np.sqrt(mean_squared_error(x, y)),
        "Bias_satellite_minus_reference": float(np.mean(y - x))
    }

# Example after creating national:
# metrics = comparison_metrics(national["GNFS_Total"], national["MODIS_Total"])
# pd.Series(metrics)


## 6. Regional comparison

In [ ]:
# regional_results = []
# for region, group in comparison.groupby("Region"):
#     stats = comparison_metrics(group["GNFS_Total"], group["MODIS_Total"])
#     regional_results.append({"Region": region, **stats})
#
# regional_summary = pd.DataFrame(regional_results).sort_values("Region")
# regional_summary


## 7. Visualise monthly national totals

In [ ]:
# fig, ax = plt.subplots(figsize=(11, 5))
# ax.plot(national["Date"], national["GNFS_Total"], marker="o",
#         label="Reference fire records")
# ax.plot(national["Date"], national["MODIS_Total"], marker="s",
#         label="MODIS active-fire counts")
# ax.set(title="Monthly Fire Counts: Reference Records and MODIS",
#        xlabel="Month", ylabel="Fire counts")
# ax.legend()
# ax.grid(alpha=0.25)
# fig.tight_layout()
# plt.show()


## 8. Reproducibility and responsible data use

- Keep raw and restricted datasets outside this public repository.
- Use relative paths and document the expected input schema.
- Record the MODIS product collection, confidence filtering, temporal window, and spatial aggregation method used in each analysis.
- Explain how missing months and unmatched records are handled.
- Do not publish GNFS data or derived tables unless you have explicit permission.
- This notebook is a cleaned, reusable workflow template based on the types of processing and comparison demonstrated in the research notebook. It is not a substitute for the full thesis methodology or a claim that the example has been run against the original data.
